# 05. Feature matching / Сопоставление особенностей

**RU.** Реальные пары фотографий Oxford Affine Covariant Features. Наблюдение — пара кадров одной сцены; признаки — координаты ключевых точек и SIFT/ORB-дескрипторы; целевая разметка — гомография H1→j. `graf` и `leuven` служат для демонстрации и настройки. `wall` и `bikes` остаются для итоговой проверки после фиксации параметров. Кадры одной сцены нельзя случайно распределять между настройкой и тестом. Это учебное разбиение четырёх сцен, не официальный benchmark и не оценка всей популяции сцен.

**EN.** Real photograph pairs from Oxford Affine Covariant Features. An observation is a pair of views of one scene. Features are keypoint coordinates and SIFT/ORB descriptors. The reference target is the H1→j homography. Use `graf` and `leuven` for demonstrations and tuning. Reserve `wall` and `bikes` for evaluation after freezing settings. Never randomly split frames of the same scene across tuning and evaluation. This four-scene teaching split is not the official benchmark or an estimate over all scenes.

**Источники / Sources:** [Oxford dataset](https://www.robots.ox.ac.uk/~vgg/research/affine/), [Data description](https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/DataREADME), Mikolajczyk et al., IJCV 2005; [OpenCV matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html).

**Условия / Terms.** Данные доступны на странице авторов для оценки методов; отдельная открытая лицензия на фотографии там не указана. Сохраняйте ссылки и авторство, не приписывайте фотографиям MIT-лицензию кода. Ноутбук загружает оригиналы с Oxford, изображения в репозиторий не включены. / The authors provide data for method evaluation; the page does not state a separate open image license. Retain attribution and sources. Do not apply a code MIT license to photographs. The notebook downloads original data from Oxford; images are not bundled in the repository.

**Ограничения / Limits.** Плоские сцены или фиксированный центр камеры. Гомография не моделирует произвольный параллакс. Эталонные H имеют погрешность. / Planar scenes or a fixed camera centre. A homography does not model arbitrary parallax. Reference homographies have uncertainty.

**Среда / Environment.** Python 3.11+, CPU, NumPy 2.2.6, OpenCV 4.12.0.88. Четыре архива около 33 MB. SHA-256 проверяется, синтетического fallback нет. / Four archives total about 33 MB. Downloads are checked with SHA-256. There is no synthetic fallback.


In [ ]:
# Colab: run once if necessary, then restart the runtime if requested.
# Colab: при необходимости выполните и перезапустите среду.
# %pip install -q numpy==2.2.6 opencv-python-headless==4.12.0.88 matplotlib


In [ ]:
DATA_MANIFEST = {'graf': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/graf.tar.gz', 'sha256': '999871b945ee968a00a0d5f9af957d1382fb9dae1511cdee9553366817b53b5b', 'bytes': 8414417}, 'leuven': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/leuven.tar.gz', 'sha256': '9b1ade6e64e5b27eee43463ff8f9ad2bd2f9c6911666e93de2634f3a5c0fe5dc', 'bytes': 6739237}, 'wall': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/wall.tar.gz', 'sha256': '3cb3df75f40471f093adf6647d13df6a9f4312f7a222fc591d03f6c08e086344', 'bytes': 9991441}, 'bikes': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/bikes.tar.gz', 'sha256': '3b95c49accadfd71e63c2fa0f7602abcbadd87d7f53ff54eb1efb15210d19e0c', 'bytes': 7366360}}
from pathlib import Path
import hashlib, io, json, tarfile, urllib.request
import numpy as np
import cv2
cv2.setNumThreads(1)
cv2.setRNGSeed(5102026)
DATA = Path('data')
MANIFEST = DATA_MANIFEST

def load_sequence(name):
    spec=MANIFEST[name]; archive=DATA/(name+'.tar.gz'); folder=DATA/name
    DATA.mkdir(exist_ok=True)
    if not archive.exists():
        with urllib.request.urlopen(spec['url'],timeout=120) as response: blob=response.read()
        if hashlib.sha256(blob).hexdigest()!=spec['sha256']: raise RuntimeError('Dataset checksum mismatch')
        archive.write_bytes(blob)
    if hashlib.sha256(archive.read_bytes()).hexdigest()!=spec['sha256']: raise RuntimeError('Dataset checksum mismatch')
    folder.mkdir(exist_ok=True)
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            if not member.isfile(): continue
            # Only known flat dataset assets, never arbitrary archive paths.
            leaf=Path(member.name).name
            if leaf not in {*(f'img{i}.ppm' for i in range(1,7)),*(f'H1to{i}p' for i in range(2,7))}:continue
            target=folder/leaf
            if not target.exists():target.write_bytes(tar.extractfile(member).read())
    return folder

def pair(sequence, index=2):
    folder=load_sequence(sequence)
    a=cv2.imread(str(folder/'img1.ppm'),cv2.IMREAD_GRAYSCALE)
    b=cv2.imread(str(folder/f'img{index}.ppm'),cv2.IMREAD_GRAYSCALE)
    if a is None or b is None:raise RuntimeError('Real dataset images unavailable')
    H=np.loadtxt(folder/f'H1to{index}p')
    # Official Oxford repeatability.m specifies coordinates 0..width.
    # Use H1toj directly with OpenCV coordinates; do not add a 1-pixel shift.
    return a,b,H/H[2,2]

def features(image,method='SIFT'):
    detector=cv2.SIFT_create(nfeatures=1500) if method=='SIFT' else cv2.ORB_create(nfeatures=1500,WTA_K=2)
    return detector.detectAndCompute(image,None)

def neighbors(d1,d2,method='SIFT'):
    if d1 is None or d2 is None or len(d1)==0 or len(d2)<2:return []
    norm=cv2.NORM_L2 if method=='SIFT' else cv2.NORM_HAMMING
    return cv2.BFMatcher(norm).knnMatch(d1,d2,k=2)

def ratio_filter(knn,ratio=.75):
    if not 0<ratio<1:raise ValueError('ratio must be between zero and one')
    return [m for row in knn if len(row)==2 for m,n in [sorted(row,key=lambda x:x.distance)] if m.distance < ratio*n.distance]

def estimate_homography(kp1,kp2,matches,threshold=3.):
    if threshold<=0:raise ValueError('positive threshold required')
    if len(matches)<4:return None,np.zeros(len(matches),dtype=bool)
    p=np.float64([kp1[m.queryIdx].pt for m in matches]);q=np.float64([kp2[m.trainIdx].pt for m in matches])
    cv2.setRNGSeed(5102026)
    H,mask=cv2.findHomography(p,q,cv2.RANSAC,threshold,maxIters=5000,confidence=.995)
    if H is None or mask is None or not np.isfinite(H).all():return None,np.zeros(len(matches),dtype=bool)
    return H,mask.ravel().astype(bool)

def transfer_errors(p,q,H):
    p=np.asarray(p,dtype=float);q=np.asarray(q,dtype=float)
    if p.shape!=q.shape or p.ndim!=2 or p.shape[1]!=2:raise ValueError('Expected paired N x 2 coordinates')
    z=np.column_stack([p,np.ones(len(p))]) @ np.asarray(H,dtype=float).T
    errors=np.full(len(p),np.inf);valid=np.isfinite(z).all(axis=1)&(np.abs(z[:,2])>1e-12)&np.isfinite(q).all(axis=1)
    errors[valid]=np.linalg.norm(z[valid,:2]/z[valid,2,None]-q[valid],axis=1)
    return errors

def choose_config(records):
    valid=[r for r in records if r['split']=='validation' and np.isfinite(r['precision'])]
    if not valid:raise ValueError('No validation records')
    ids=sorted({r['config'] for r in valid})
    # Macro-average over image pairs, then lexicographic tie break.
    return min(ids,key=lambda c:(-float(np.mean([r['precision'] for r in valid if r['config']==c])),c))

def evaluate(sequence,index,method='SIFT',ratio=.75):
    a,b,gt=pair(sequence,index);k1,d1=features(a,method);k2,d2=features(b,method)
    good=ratio_filter(neighbors(d1,d2,method),ratio);H,inliers=estimate_homography(k1,k2,good)
    p=np.float64([k1[m.queryIdx].pt for m in good]).reshape(-1,2);q=np.float64([k2[m.trainIdx].pt for m in good]).reshape(-1,2)
    e=transfer_errors(p,q,gt); precision=float(np.mean(e<=3)) if len(e) else 0.
    metric=dict(sequence=sequence,pair=f'1-{index}',method=method,ratio=ratio,matches=len(good),inliers=int(inliers.sum()),precision=precision,gt_median_px=float(np.median(e)) if len(e) else None,homography_found=H is not None)
    return metric,(a,b,k1,k2,good,H,inliers,gt)


## 1. Две реальные фотографии / Two real photographs
RU: осмотрите граффити на кадрах 1 и 2. Назовите изменение точки зрения, повторяющиеся структуры и возможные неоднозначности.
EN: inspect graffiti frames 1 and 2. Identify viewpoint changes, repeated structures and possible ambiguities.

In [ ]:
a,b,H_gt=pair('graf',2)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,img in zip(axes,[a,b]): ax.imshow(img,cmap='gray'); ax.axis('off')
plt.show()

## 2. SIFT, ORB и расстояния / SIFT, ORB and distances
RU: SIFT использует L2, ORB с WTA_K=2 — Hamming. Ratio test сравнивает двух соседей. Число соответствий само по себе не измеряет правильность.
EN: use L2 for SIFT and Hamming for ORB with WTA_K=2. The ratio test compares two neighbours. Match count alone does not measure correctness.

In [ ]:
for method in ['SIFT','ORB']:
    m,aux=evaluate('graf',2,method,.75)
    print(m)
    a,b,k1,k2,good,H,inliers,gt=aux
    image=cv2.drawMatches(a,k1,b,k2,good[:40],None,flags=2)
    plt.figure(figsize=(14,6));plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()

## 3. Проверка геометрии / Geometric verification
RU: маска RANSAC проверяет согласованность с найденной моделью. Precision@3px здесь использует независимую H из датасета. Это разные проверки. Ошибка — расстояние между H_gt p и q, в пикселях второго исходного изображения. Оценка H не получает H_gt на вход.
EN: the RANSAC mask tests consistency with the estimated model. Teaching precision@3px uses the independent dataset homography. These are different tests. The error is the distance from H_gt p to q in original target-image pixels. Homography estimation never receives H_gt.

In [ ]:
m,aux=evaluate('graf',4,'SIFT',.75)
a,b,k1,k2,good,H,mask,gt=aux
print(m)
if H is not None:
    aligned=cv2.warpPerspective(a,H,(b.shape[1],b.shape[0]))
    plt.figure(figsize=(10,6));plt.imshow(cv2.addWeighted(aligned,.5,b,.5,0),cmap='gray');plt.axis('off');plt.show()
else:
    print('Гомография не найдена / Homography not found')

## 4. Настройка и отложенные сцены / Tuning and held-out scenes
RU: выбираем по среднему precision@3px на четырёх validation-парах. Нулевые соответствия дают precision=0; всегда выводим число matches, чтобы один удачный match не выглядел убедительно. При равенстве — лексикографически минимальный ID. Фиксируем выбор до открытия test. После test параметры не меняем.
EN: select by mean precision@3px over four validation pairs. Zero matches yield zero precision. Always report match counts so a single correct match does not look persuasive. Break ties by lexicographic configuration ID. Freeze before opening test. Do not retune after test.

In [ ]:
records=[]
for seq in ['graf','leuven']:
    for method in ['SIFT','ORB']:
        for ratio in [.6,.75,.9]:
            for index in [2,4]:
                r,_=evaluate(seq,index,method,ratio)
                r.update(split='validation',config=f'{method}:{ratio}');records.append(r)
selected=choose_config(records)
Path('frozen_config.json').write_text(json.dumps({'selected':selected}))
print('Frozen:',selected)

In [ ]:
selected=json.loads(Path('frozen_config.json').read_text())['selected']
method,ratio=selected.split(':')
for seq in ['wall','bikes']:
    for index in [2,4,6]:
        r,_=evaluate(seq,index,method,float(ratio));r.update(split='test',config=selected);records.append(r);print(r)
Path('matching_results.json').write_text(json.dumps(records,indent=2))

## Четыре задания на занятии / Four in-class tasks
1. **Дескрипторы / Descriptors (10 мин / min).** На `graf` 1–2 сравните SIFT и ORB: размерность, dtype, норму и число соответствий. Объясните выбор нормы. / Compare dimensions, dtype, distance norm and match counts on `graf` 1–2. Explain each norm.
2. **Ratio test (15 мин / min).** На validation-парах сравните 0.6, 0.75 и 0.9. Покажите количество и precision@3px. Объясните компромисс, не выбирая по test. / Compare the three thresholds on validation pairs. Report counts and precision. Explain the trade-off without tuning on test.
3. **RANSAC (15 мин / min).** На `graf` 1–4 сравните пороги 1, 3 и 6 px. Выведите маску inliers и независимую ошибку относительно H_gt. / Compare thresholds of 1, 3 and 6 px on `graf` 1–4. Show inliers and independent ground-truth error.
4. **Отложенная оценка / Held-out evaluation (15 мин / min).** Зафиксируйте конфигурацию, оцените `wall` и `bikes`, объясните один провал и ограничение вывода по двум сценам. / Freeze configuration, evaluate `wall` and `bikes`, explain a failure and the limits of inference from two scenes.

Сдайте ноутбук с выводами и `matching_results.json`. Отметку практики ставит преподаватель. / Submit the executed notebook with explanations and results JSON. The instructor records practice completion.

## Переиспользовано и обновлено / Reuse and updates
RU: из авторского курса сохранены SIFT, ORB, matching и выравнивание. Обновление 2026: реальные серии Oxford, проверка загрузки, независимая геометрическая метрика, разделение сцен и явный отказ при нехватке matches. LightGlue рассматриваем в лекции как факультативное продолжение.
EN: the instructor's SIFT, ORB, matching and alignment topics are retained. The 2026 update adds real Oxford sequences, verified downloads, independent geometry metrics, scene separation and explicit failure handling. LightGlue is an optional lecture extension.
